In [2]:
import sys
from pathlib import Path

PROJECT_DIR = Path(r"D:\AI_Customer_Support_Agent")

print("Python executable:", sys.executable)
print("Project folder exists:", PROJECT_DIR.exists())

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}")


Python executable: d:\AI_Customer_Support_Agent\.venv\Scripts\python.exe
Project folder exists: True


In [3]:
import langchain
import langgraph
import faiss
import dotenv

print("Everything is ready!")

Everything is ready!


In [4]:
import os
from dotenv import load_dotenv

load_dotenv(
    dotenv_path=PROJECT_DIR / ".env",
    override=False
)

api_key = os.getenv("COHERE_API_KEY")

print("Cohere API key loaded:", bool(api_key))

if not api_key:
    raise ValueError(
        "COHERE_API_KEY was not found. Check your project .env file."
    )

Cohere API key loaded: True


In [5]:
faq_file = PROJECT_DIR / "data" / "faq.txt"

if not faq_file.exists():
    raise FileNotFoundError(f"FAQ file not found: {faq_file}")

faq_text = faq_file.read_text(encoding="utf-8")

print("FAQ file loaded:", faq_file)
print("FAQ characters:", len(faq_text))
print("\nFAQ preview:")
print(faq_text[:500])

FAQ file loaded: D:\AI_Customer_Support_Agent\data\faq.txt
FAQ characters: 1432

FAQ preview:
CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

Refunds are normally processed wit


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = text_splitter.create_documents([faq_text])

print("Number of FAQ chunks:", len(documents))

if not documents:
    raise ValueError("No FAQ chunks were created.")

print("\nFirst FAQ chunk:")
print(documents[0].page_content)

Number of FAQ chunks: 4

First FAQ chunk:
CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy


In [7]:
from langchain_cohere import ChatCohere
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("Cohere_api_key")

llm = ChatCohere(
    model="command-r-08-2024",
    cohere_api_key=api_key,
     temperature=0
)

In [8]:

from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=api_key
)

print("Cohere embeddings created:", embeddings is not None)

Cohere embeddings created: True


In [9]:
from langchain_community.vectorstores import FAISS

if not documents:
    raise ValueError("FAQ documents are empty.")

vector_store = FAISS.from_documents(
    documents,
    embeddings
)

print("FAISS vector store created successfully.")
print("Indexed vectors:", vector_store.index.ntotal)

C:\Users\dell\AppData\Local\Temp\ipykernel_13996\2374032102.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS vector store created successfully.
Indexed vectors: 4


In [10]:
query = "How long does a refund take?"

results = vector_store.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)


--- Result 1 ---
3. Refund Policy

Refunds are normally processed within 5 to 7 business days after the returned product is received.
The refund will be sent to the original payment method.

4. Damaged Product

If a product arrives damaged, the customer should contact customer support within 48 hours.
The customer should provide the order number and photographs of the damaged product.
The support team will review the request and arrange a replacement or refund when applicable.

5. Delivery

--- Result 2 ---
5. Delivery

Standard delivery normally takes 3 to 5 business days.
Delivery times can vary depending on the customer's location and shipping conditions.

6. Change Delivery Address

Customers can change their delivery address before the order is shipped.
After shipment, the address usually cannot be changed.

7. Contact Customer Support


In [11]:
from langchain_core.tools import tool

@tool
def knowledge_base_search(query: str) -> str:
    """Search the customer support FAQ knowledge base for an answer."""

    results = vector_store.similarity_search(query, k=2)

    if not results:
        return "No relevant information was found in the knowledge base."

    return "\n\n".join(
        doc.page_content
        for doc in results
    )

print("Knowledge Base Search Tool created")

Knowledge Base Search Tool created


In [12]:
result = knowledge_base_search.invoke(
    "How long does a refund take?"
)

print(result)


3. Refund Policy

Refunds are normally processed within 5 to 7 business days after the returned product is received.
The refund will be sent to the original payment method.

4. Damaged Product

If a product arrives damaged, the customer should contact customer support within 48 hours.
The customer should provide the order number and photographs of the damaged product.
The support team will review the request and arrange a replacement or refund when applicable.

5. Delivery

5. Delivery

Standard delivery normally takes 3 to 5 business days.
Delivery times can vary depending on the customer's location and shipping conditions.

6. Change Delivery Address

Customers can change their delivery address before the order is shipped.
After shipment, the address usually cannot be changed.

7. Contact Customer Support


In [13]:
orders = {
    "12345": {
        "status": "Shipped",
        "expected_delivery": "10 October 2026"
    },
    "12346": {
        "status": "Processing",
        "expected_delivery": "12 October 2026"
    },
    "12347": {
        "status": "Delivered",
        "expected_delivery": "5 October 2026"
    },
    "12348": {
        "status": "Cancelled",
        "expected_delivery": "N/A"
    }
}

print("Mock order database created")

Mock order database created


In [14]:
from langchain_core.tools import tool

@tool
def order_status_lookup(order_id: str) -> str:
    """Look up the status and expected delivery date of a customer order."""

    if order_id not in orders:
        return f"Order {order_id} was not found."

    order = orders[order_id]

    return (
        f"Order {order_id} status: {order['status']}. "
        f"Expected delivery: {order['expected_delivery']}."
    )

print("Order Status Tool created")

Order Status Tool created


In [15]:
result = order_status_lookup.invoke("12345")

print(result)

Order 12345 status: Shipped. Expected delivery: 10 October 2026.


In [16]:
from langchain_core.tools import tool

@tool
def escalate_to_human(customer_issue: str, order_id: str = "Not provided") -> str:
    """Draft an escalation email for a customer issue that needs human support."""

    email = f"""
Subject: Customer Support Escalation - Order {order_id}

Dear Human Support Team,

A customer needs assistance with the following issue:

{customer_issue}

Order ID: {order_id}

Please review this issue and contact the customer with the appropriate resolution.

Regards,
AI Customer Support Agent
"""

    return email.strip()

print("Escalation Tool created")

Escalation Tool created


In [17]:
result = escalate_to_human.invoke({
    "customer_issue": "My product arrived damaged and I need a replacement.",
    "order_id": "12345"
})

print(result)

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

My product arrived damaged and I need a replacement.

Order ID: 12345

Please review this issue and contact the customer with the appropriate resolution.

Regards,
AI Customer Support Agent


In [18]:
from langchain_core.tools import tool

@tool
def knowledge_base_search(query: str) -> str:
    """Search the customer support FAQ knowledge base for an answer."""

    results = vector_store.similarity_search(query, k=2)

    if not results:
        return "No relevant information found."

    return "\n\n".join(
        doc.page_content for doc in results
    )

print("Knowledge Base Tool ready")

Knowledge Base Tool ready


In [19]:
from langchain_core.tools import tool

@tool
def order_status_lookup(order_id: str) -> str:
    """Look up the status and expected delivery date of a customer order."""

    if order_id not in orders:
        return f"Order {order_id} was not found."

    order = orders[order_id]

    return (
        f"Order {order_id} status: {order['status']}. "
        f"Expected delivery: {order['expected_delivery']}."
    )

print("Order Status Tool created successfully")

Order Status Tool created successfully


In [20]:
from langchain_core.tools import tool

@tool
def escalate_to_human(
    customer_issue: str,
    order_id: str = "Not provided"
) -> str:
    """Draft an escalation email for a customer issue requiring human support."""

    email = f"""
Subject: Customer Support Escalation - Order {order_id}

Dear Human Support Team,

Customer issue:
{customer_issue}

Order ID: {order_id}

Please investigate this issue and contact the customer.

Regards,
AI Customer Support Agent
"""

    return email.strip()

print("Escalation Tool created successfully")

Escalation Tool created successfully


In [21]:
tools = [
    knowledge_base_search,
    order_status_lookup,
    escalate_to_human
]

print("Total tools:", len(tools))

for t in tools:
    print(t.name)

Total tools: 3
knowledge_base_search
order_status_lookup
escalate_to_human


In [22]:
from langchain.agents import create_agent
from langchain_cohere import ChatCohere
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatCohere(
    model="command-r-08-2024",
    temperature=0,
    cohere_api_key=os.getenv("COHERE_API_KEY")
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful AI customer support agent.

    - Use knowledge_base_search for FAQ questions.
    - Use order_status_lookup for order status questions.
    - Use escalate_to_human for complaints requiring human assistance.
    - Never invent order information.
    - Give polite, clear answers.
    """
)

print("Cohere AI Customer Support Agent created successfully")


Cohere AI Customer Support Agent created successfully


In [23]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents, embeddings)

print("Vector store created successfully!")


Vector store created successfully!


In [24]:
from pathlib import Path

faq_file = Path(r"D:\AI_Customer_Support_Agent\data\faq.txt")

faq_text = faq_file.read_text(encoding="utf-8")

print("FAQ file loaded successfully!")
print(faq_text[:300])

FAQ file loaded successfully!
CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before


In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = text_splitter.create_documents([faq_text])

print("Documents created:", len(documents))

Documents created: 4


In [26]:
from langchain_cohere import CohereEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()

embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=os.getenv("COHERE_API_KEY")
)

print("Cohere embeddings created successfully!")

Cohere embeddings created successfully!


In [27]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(documents, embeddings)

print("Vector store created successfully!")

Vector store created successfully!


In [28]:
results = vector_store.similarity_search(
    "How long does a refund take?",
    k=2
)

for result in results:
    print(result.page_content)
    print("--------------------")

3. Refund Policy

Refunds are normally processed within 5 to 7 business days after the returned product is received.
The refund will be sent to the original payment method.

4. Damaged Product

If a product arrives damaged, the customer should contact customer support within 48 hours.
The customer should provide the order number and photographs of the damaged product.
The support team will review the request and arrange a replacement or refund when applicable.

5. Delivery
--------------------
5. Delivery

Standard delivery normally takes 3 to 5 business days.
Delivery times can vary depending on the customer's location and shipping conditions.

6. Change Delivery Address

Customers can change their delivery address before the order is shipped.
After shipment, the address usually cannot be changed.

7. Contact Customer Support
--------------------


In [29]:
query = "How long does a refund take?"

results = vector_store.similarity_search(query, k=2)

for result in results:
    print(result.page_content)
    print("--------------------")

3. Refund Policy

Refunds are normally processed within 5 to 7 business days after the returned product is received.
The refund will be sent to the original payment method.

4. Damaged Product

If a product arrives damaged, the customer should contact customer support within 48 hours.
The customer should provide the order number and photographs of the damaged product.
The support team will review the request and arrange a replacement or refund when applicable.

5. Delivery
--------------------
5. Delivery

Standard delivery normally takes 3 to 5 business days.
Delivery times can vary depending on the customer's location and shipping conditions.

6. Change Delivery Address

Customers can change their delivery address before the order is shipped.
After shipment, the address usually cannot be changed.

7. Contact Customer Support
--------------------


In [30]:
user_input = "How long does a refund take?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": user_input}
    ]
})

print(response["messages"][-1].content)

Refunds are normally processed within 5 to 7 business days after the returned product is received. The refund will be sent to the original payment method.


In [31]:
print(order_status_lookup.name)
print(orders)

order_status_lookup
{'12345': {'status': 'Shipped', 'expected_delivery': '10 October 2026'}, '12346': {'status': 'Processing', 'expected_delivery': '12 October 2026'}, '12347': {'status': 'Delivered', 'expected_delivery': '5 October 2026'}, '12348': {'status': 'Cancelled', 'expected_delivery': 'N/A'}}


In [32]:
orders = {
    "12345": {
        "status": "Shipped",
        "expected_delivery": "10 October 2026"
    },
    "12346": {
        "status": "Processing",
        "expected_delivery": "12 October 2026"
    },
    "12347": {
        "status": "Delivered",
        "expected_delivery": "5 October 2026"
    },
    "12348": {
        "status": "Cancelled",
        "expected_delivery": "N/A"
    }
}

print("Order database created successfully!")
print(orders)

Order database created successfully!
{'12345': {'status': 'Shipped', 'expected_delivery': '10 October 2026'}, '12346': {'status': 'Processing', 'expected_delivery': '12 October 2026'}, '12347': {'status': 'Delivered', 'expected_delivery': '5 October 2026'}, '12348': {'status': 'Cancelled', 'expected_delivery': 'N/A'}}


In [33]:
user_input = "Where is my order 12345?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": user_input}
    ]
})

print(response["messages"][-1].content)

Your order 12345 has been shipped and is expected to arrive on 10 October 2026.


In [34]:
from langchain_core.tools import tool

@tool
def escalate_to_human(
    customer_issue: str,
    order_id: str = "Not provided"
) -> str:
    """Draft an escalation email for a customer issue that needs human support."""

    email = f"""
Subject: Customer Support Escalation - Order {order_id}

Dear Human Support Team,

A customer needs assistance with the following issue:

{customer_issue}

Order ID: {order_id}

Please review this issue and contact the customer
with the appropriate resolution.

Regards,
AI Customer Support Agent
"""

    return email.strip()

print("Escalation tool created successfully!")

Escalation tool created successfully!


In [35]:
result = escalate_to_human.invoke({
    "customer_issue": "The product arrived damaged.",
    "order_id": "12345"
})

print(result)

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

The product arrived damaged.

Order ID: 12345

Please review this issue and contact the customer
with the appropriate resolution.

Regards,
AI Customer Support Agent


In [36]:
user_input = (
    "My order 12345 arrived damaged. "
    "Please escalate this issue to human support."
)

response = agent.invoke({
    "messages": [
        {"role": "user", "content": user_input}
    ]
})

print(response["messages"][-1].content)

Here is a draft of an email to escalate the issue to human support:

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

Customer issue:
My order 12345 arrived damaged.

Order ID: 12345

Please investigate this issue and contact the customer.

Regards,
AI Customer Support Agent


In [37]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful AI customer support agent.

1. Use knowledge_base_search for FAQ questions.
2. Use order_status_lookup for order-status questions.
3. Use escalate_to_human for complaints requiring human assistance.
4. Never invent order information.
5. Give polite, clear answers.
6. Use previous messages in the same conversation thread for follow-up questions.
7. If an order ID is available in the conversation history, use it when relevant.
8. Ask for the order ID only if it is not available in the conversation history.
9. The escalation tool drafts an email; do not claim that an email was sent.
10. Use the FAQ knowledge base for FAQ answers.
""",
    checkpointer=memory
)

print("Agent created:", agent is not None)
print("Memory created:", memory is not None)

Agent created: True
Memory created: True


In [38]:
config = {
    "configurable": {
        "thread_id": "customer-123"
    }
}

response1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "My order number is 12345."}
        ]
    },
    config=config
)

print(response1["messages"][-1].content)

Thank you for providing your order number. I can confirm that your order has been shipped and is expected to arrive on 10 October 2026.


In [39]:
response2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is its current status?"}
        ]
    },
    config=config
)

print(response2["messages"][-1].content)

The current status of your order is "Shipped".


In [40]:
user_input = (
    "My order 12345 arrived damaged. "
    "Please escalate this issue to human support."
)

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_input}
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Here is a draft of the email I have prepared to escalate this issue to human support:

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

Customer issue:
My order 12345 arrived damaged.

Order ID: 12345

Please investigate this issue and contact the customer.

Regards,
AI Customer Support Agent


In [41]:
for t in tools:
    print("Tool:", t.name)

Tool: knowledge_base_search
Tool: order_status_lookup
Tool: escalate_to_human


In [42]:
user_input = "How can I reset my password?"

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_input}
        ]
    },
    config={
        "configurable": {
            "thread_id": "faq-test-001"
        }
    }
)

print(response["messages"][-1].content)

To reset your password, you need to click "Forgot Password" on the login page. You will then receive a password reset link on your registered email address. The reset link is valid for 30 minutes.


In [43]:
query = "How can I reset my password?"

results = vector_store.similarity_search(query, k=2)

for result in results:
    print(result.page_content)
    print("--------------------")

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy
--------------------
7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.
--------------------


In [44]:
result = knowledge_base_search.invoke({
    "query": "How can I reset my password?"
})

print(result)

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.


In [45]:
query = "How can I reset my password?"

results = vector_store.similarity_search(query, k=2)

for result in results:
    print(result.page_content)
    print("--------------------")

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy
--------------------
7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.
--------------------


In [46]:
result = knowledge_base_search.invoke({
    "query": "How can I reset my password?"
})

print(result)

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.


In [47]:
print("AI CUSTOMER SUPPORT AGENT")
print("=========================")

for tool_item in tools:
    print("Tool:", tool_item.name)

print("Total tools:", len(tools))

AI CUSTOMER SUPPORT AGENT
Tool: knowledge_base_search
Tool: order_status_lookup
Tool: escalate_to_human
Total tools: 3


In [48]:
project_report = {
    "Project": "AI Customer Support Agent",
    "FAQ Knowledge Base (FAISS)": "Tested",
    "Order Status Lookup": "Tested",
    "Human Escalation Tool": "Tested",
    "Conversation Memory": "Tested",
    "Total Tools": len(tools),
    "Tool Names": [tool_item.name for tool_item in tools]
}

for key, value in project_report.items():
    print(f"{key}: {value}")

Project: AI Customer Support Agent
FAQ Knowledge Base (FAISS): Tested
Order Status Lookup: Tested
Human Escalation Tool: Tested
Conversation Memory: Tested
Total Tools: 3
Tool Names: ['knowledge_base_search', 'order_status_lookup', 'escalate_to_human']


In [49]:
tools = [
    knowledge_base_search,
    order_status_lookup,
    escalate_to_human
]

print("Total tools:", len(tools))

for tool_item in tools:
    print("Tool:", tool_item.name)

Total tools: 3
Tool: knowledge_base_search
Tool: order_status_lookup
Tool: escalate_to_human


In [50]:
project_report = {
    "Project": "AI Customer Support Agent",
    "FAQ Knowledge Base (FAISS)": "Tested",
    "Order Status Lookup": "Tested",
    "Human Escalation Tool": "Tested",
    "Conversation Memory": "Tested",
    "Total Tools": len(tools),
    "Tool Names": [tool_item.name for tool_item in tools]
}

for key, value in project_report.items():
    print(f"{key}: {value}")

Project: AI Customer Support Agent
FAQ Knowledge Base (FAISS): Tested
Order Status Lookup: Tested
Human Escalation Tool: Tested
Conversation Memory: Tested
Total Tools: 3
Tool Names: ['knowledge_base_search', 'order_status_lookup', 'escalate_to_human']


In [51]:
print("===== AI CUSTOMER SUPPORT AGENT =====")

print("\n1. FAQ Knowledge Base")
result = knowledge_base_search.invoke({
    "query": "How can I reset my password?"
})
print(result)

print("\n2. Order Status Lookup")
result = order_status_lookup.invoke({
    "order_id": "12345"
})
print(result)

print("\n3. Human Escalation")
result = escalate_to_human.invoke({
    "customer_issue": "The product arrived damaged.",
    "order_id": "12345"
})
print(result)

print("\n===== DEMONSTRATION COMPLETED =====")

===== AI CUSTOMER SUPPORT AGENT =====

1. FAQ Knowledge Base
CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.

2. Order Status Lookup
Order 12345 status: Shipped. Expected delivery: 10 October 2026.

3. Human Escalation
Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

The product arrived dam

In [52]:
test_cases = [
    {
        "name": "FAQ Search",
        "tool": knowledge_base_search,
        "input": {"query": "How can I reset my password?"}
    },
    {
        "name": "Order Status",
        "tool": order_status_lookup,
        "input": {"order_id": "12345"}
    },
    {
        "name": "Human Escalation",
        "tool": escalate_to_human,
        "input": {
            "customer_issue": "The product arrived damaged.",
            "order_id": "12345"
        }
    }
]

for test in test_cases:
    print("\nTEST:", test["name"])
    result = test["tool"].invoke(test["input"])
    print(result)
    print("-" * 50)


TEST: FAQ Search
CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.
--------------------------------------------------

TEST: Order Status
Order 12345 status: Shipped. Expected delivery: 10 October 2026.
--------------------------------------------------

TEST: Human Escalation
Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs ass

In [53]:
query = "How can I reset my password?"

results = vector_store.similarity_search(query, k=3)

for result in results:
    print(result.page_content)
    print("-" * 50)

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy
--------------------------------------------------
7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.
--------------------------------------------------
3. Refund Policy

Refunds are normally processed within 5 to 7 business days after the returned product is received.
The refund will be sent to the original payment method.

4. Damaged Product

If a product arrives damaged, th

In [54]:
user_input = "How can I reset my password?"

response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_input}
        ]
    },
    config={
        "configurable": {
            "thread_id": "faq-final-test"
        }
    }
)

print(response["messages"][-1].content)

To reset your password, you need to click 'Forgot Password' on the login page. You will then receive a password reset link on your registered email address. This link is valid for 30 minutes.


In [55]:
print(response["messages"][-1].text)

To reset your password, you need to click 'Forgot Password' on the login page. You will then receive a password reset link on your registered email address. This link is valid for 30 minutes.


In [56]:
user_input = "Where is my order 12345?"

response = agent.invoke(
    {"messages": [{"role": "user", "content": user_input}]},
    config={"configurable": {"thread_id": "order-test-001"}}
)

print(response["messages"][-1].text)

Your order is currently on its way to you. It was shipped on 10 October 2026 and you should receive it soon.


In [57]:
print("Knowledge Base Tool:", knowledge_base_search.name)
print("Order Status Tool:", order_status_lookup.name)
print("Escalation Tool:", escalate_to_human.name)

Knowledge Base Tool: knowledge_base_search
Order Status Tool: order_status_lookup
Escalation Tool: escalate_to_human


In [58]:
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

print("Memory system:", type(memory).__name__) 
print("Agent ready:", agent is not None)

Memory system: InMemorySaver
Agent ready: True


In [59]:
print("Agent ready:", agent is not None)
print("Tools available:", len(tools))
print("Memory system:", type(memory).__name__)

Agent ready: True
Tools available: 3
Memory system: InMemorySaver


In [60]:
result = knowledge_base_search.invoke({
    "query": "How can I reset my password?"
})

print(result)

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

7. Contact Customer Support

Customers can contact the support team when their issue cannot be resolved using the FAQ.
For urgent or unresolved issues, the request can be escalated to a human support agent.


In [61]:
result = order_status_lookup.invoke({
    "order_id": "12345"
})

print(result)

Order 12345 status: Shipped. Expected delivery: 10 October 2026.


In [62]:
result = escalate_to_human.invoke({
    "customer_issue": "My product arrived damaged.",
    "order_id": "12345"
})

print(result)

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

My product arrived damaged.

Order ID: 12345

Please review this issue and contact the customer
with the appropriate resolution.

Regards,
AI Customer Support Agent


In [63]:
from pathlib import Path

project_path = Path(r"D:\AI_Customer_Support_Agent")

print("Project folder exists:", project_path.exists())
print("FAQ file exists:", (project_path / "data" / "faq.txt").exists())
print("Notebook project path:", project_path)

Project folder exists: True
FAQ file exists: True
Notebook project path: D:\AI_Customer_Support_Agent


In [64]:
from pathlib import Path

faq_file = Path(r"D:\AI_Customer_Support_Agent\data\faq.txt")

print(faq_file.exists())
print(faq_file)

True
D:\AI_Customer_Support_Agent\data\faq.txt


In [65]:
print(faq_file.exists())
print(faq_file)

True
D:\AI_Customer_Support_Agent\data\faq.txt


In [66]:
print(faq_text[:500])

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy

Refunds are normally processed wit


In [67]:
print("Total FAQ chunks:", len(documents))
print("First chunk:")
print(documents[0].page_content)

Total FAQ chunks: 4
First chunk:
CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy


In [68]:
print("FAISS knowledge base ready:", vector_store is not None)
print("Documents indexed:", vector_store.index.ntotal)

FAISS knowledge base ready: True
Documents indexed: 4


In [69]:
result = knowledge_base_search.invoke({
    "query": "What is the refund policy?"
})

print(result)

3. Refund Policy

Refunds are normally processed within 5 to 7 business days after the returned product is received.
The refund will be sent to the original payment method.

4. Damaged Product

If a product arrives damaged, the customer should contact customer support within 48 hours.
The customer should provide the order number and photographs of the damaged product.
The support team will review the request and arrange a replacement or refund when applicable.

5. Delivery

CUSTOMER SUPPORT FAQ

1. Password Reset

Customers can reset their password by clicking "Forgot Password" on the login page.
They will receive a password reset link on their registered email address.
The reset link is valid for 30 minutes.

2. Order Cancellation

Customers can cancel an order before it is shipped.
Once an order has been shipped, it cannot normally be cancelled.
Customers should contact support if they need further assistance.

3. Refund Policy


In [70]:
result = order_status_lookup.invoke({
    "order_id": "12346"
})

print(result)

Order 12346 status: Processing. Expected delivery: 12 October 2026.


In [71]:
result = escalate_to_human.invoke({
    "customer_issue": "My product arrived damaged.",
    "order_id": "12345"
})

print(result)

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

My product arrived damaged.

Order ID: 12345

Please review this issue and contact the customer
with the appropriate resolution.

Regards,
AI Customer Support Agent


In [72]:
print("Memory system:", type(memory).__name__)
print("Agent ready:", agent is not None)
print("Total tools:", len(tools))

Memory system: InMemorySaver
Agent ready: True
Total tools: 3


In [73]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

faq_file = Path(r"D:\AI_Customer_Support_Agent\data\faq.txt")

faq_text = faq_file.read_text(encoding="utf-8")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = text_splitter.create_documents([faq_text])

vector_store = FAISS.from_documents(documents, embeddings)

print("Cohere FAISS vector store created successfully!")
print("Number of FAQ chunks:", len(documents))

Cohere FAISS vector store created successfully!
Number of FAQ chunks: 4


In [74]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

faq_file = Path(r"D:\AI_Customer_Support_Agent\data\faq.txt")

faq_text = faq_file.read_text(encoding="utf-8")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

documents = text_splitter.create_documents([faq_text])

vector_store = FAISS.from_documents(documents, embeddings)

print("Cohere FAISS vector store created successfully!")
print("Number of FAQ chunks:", len(documents))

Cohere FAISS vector store created successfully!
Number of FAQ chunks: 4


In [75]:
print("Available tools:")

for tool in tools:
    print("-", tool.name)

Available tools:
- knowledge_base_search
- order_status_lookup
- escalate_to_human


In [76]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful AI customer support agent.

1. Use knowledge_base_search for FAQ questions.
2. Use order_status_lookup for order-status questions.
3. Use escalate_to_human for complaints requiring human assistance.
4. Never invent order information.
5. Give polite, clear answers.
6. Use previous messages in the same conversation thread for follow-up questions.
7. If an order ID is available in the conversation history, use it when relevant.
8. Ask for the order ID only if it is not available in the conversation history.
9. The escalation tool drafts an email; do not claim that an email was sent.
10. Use the FAQ knowledge base for FAQ answers.
""",
    checkpointer=memory
)

print("Agent created:", agent is not None)
print("Memory created:", memory is not None)

Agent created: True
Memory created: True


In [77]:
config = {
    "configurable": {
        "thread_id": "faq-demo-001"
    }
}

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is your refund policy?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Refunds are normally processed within 5 to 7 business days after the returned product is received. The refund will be sent to the original payment method.


In [78]:

config = {
    "configurable": {
        "thread_id": "order-demo-001"
    }
}

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the status of order 12345?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)


Your order has been shipped and is expected to arrive on 10 October 2026.


In [79]:

config = {
    "configurable": {
        "thread_id": "escalation-demo-001"
    }
}

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My order arrived damaged. Please escalate this complaint to human support. Order ID is 12345."
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)


Thank you for bringing this to our attention. I have drafted an email to escalate your issue to our human support team.

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

My order arrived damaged.

Order ID: 12345

Please review this issue and contact the customer
with the appropriate resolution.

Regards,
AI Customer Support Agent


In [80]:

config = {
    "configurable": {
              "thread_id": "order-demo-001"
    }
}

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "When is my order expected to arrive?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)


Your order is expected to arrive on 10 October 2026.


In [81]:

config = {
    "configurable": {
        "thread_id": "order-demo-001"
    }
}

history = agent.get_state(config)

print("Conversation memory details:")

if history and history.values:
    messages = history.values.get("messages", [])

    for message in messages:
        print(f"{message.type}: {message.content}")
else:
    print("No conversation history found.")


Conversation memory details:
human: What is the status of order 12345?
ai: I will search for the order status of order 12345.
tool: Order 12345 status: Shipped. Expected delivery: 10 October 2026.
ai: Your order has been shipped and is expected to arrive on 10 October 2026.
human: When is my order expected to arrive?
ai: I will search for the order status of order 12345.
tool: Order 12345 status: Shipped. Expected delivery: 10 October 2026.
ai: Your order is expected to arrive on 10 October 2026.


In [82]:
print("AI Customer Support Agent - Tool Check")

print("\nAvailable tools:")
for tool in tools:
    print("-", tool.name)

print("\nAgent created:", agent is not None)
print("Conversation memory created:", memory is not None)
print("FAQ vector store created:", vector_store is not None)

AI Customer Support Agent - Tool Check

Available tools:
- knowledge_base_search
- order_status_lookup
- escalate_to_human

Agent created: True
Conversation memory created: True
FAQ vector store created: True


In [83]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Check the status of order 99999."
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

I'm sorry, I couldn't find any information on order 99999.


In [84]:
project_summary = {
    "Project Name": "AI Customer Support Agent",
    "LLM": "Cohere ChatCohere",
    "Embeddings": "Cohere Embeddings",
    "Vector Database": "FAISS",
    "Knowledge Base": "FAQ text file",
    "Tools": [
        "knowledge_base_search",
        "order_status_lookup",
        "escalate_to_human"
    ],
    "Memory": "LangGraph InMemorySaver",
    "Framework": "LangChain and LangGraph"
}

for key, value in project_summary.items():
    print(f"{key}: {value}")

Project Name: AI Customer Support Agent
LLM: Cohere ChatCohere
Embeddings: Cohere Embeddings
Vector Database: FAISS
Knowledge Base: FAQ text file
Tools: ['knowledge_base_search', 'order_status_lookup', 'escalate_to_human']
Memory: LangGraph InMemorySaver
Framework: LangChain and LangGraph


In [85]:
print("===== AI CUSTOMER SUPPORT AGENT =====")

print("1. Cohere LLM:", llm is not None)
print("2. FAQ vector store:", vector_store is not None)
print("3. Agent:", agent is not None)
print("4. Conversation memory:", memory is not None)

print("\nAvailable tools:")
for tool in tools:
    print("-", tool.name)

print("\nProject component check completed!")

===== AI CUSTOMER SUPPORT AGENT =====
1. Cohere LLM: True
2. FAQ vector store: True
3. Agent: True
4. Conversation memory: True

Available tools:
- knowledge_base_search
- order_status_lookup
- escalate_to_human

Project component check completed!


In [86]:
print("===== AI CUSTOMER SUPPORT AGENT =====")

print("1. Cohere LLM:", llm is not None)
print("2. FAQ vector store:", vector_store is not None)
print("3. Agent:", agent is not None)
print("4. Conversation memory:", memory is not None)

print("\nAvailable tools:")
for tool in tools:
    print("-", tool.name)

print("\nProject component check completed!")

===== AI CUSTOMER SUPPORT AGENT =====
1. Cohere LLM: True
2. FAQ vector store: True
3. Agent: True
4. Conversation memory: True

Available tools:
- knowledge_base_search
- order_status_lookup
- escalate_to_human

Project component check completed!


In [87]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My order 12345 has not arrived. Can you check its status and help me?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

I can see that your order has been shipped and is expected to arrive on 10 October 2026. If you have not received it by then, please get in touch and we can investigate further.


In [88]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is your refund policy?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Refunds are normally processed within 5 to 7 business days after the returned product is received. The refund will be sent to the original payment method.


In [89]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My order arrived damaged. Please escalate this complaint to human support. Order ID is 12345."
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)


Here is a draft of the email I have prepared to escalate this issue to the human support team:

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

The order arrived damaged.

Order ID: 12345

Please review this issue and contact the customer with the appropriate resolution.

Regards,
AI Customer Support Agent


In [90]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My order arrived damaged. Please escalate this complaint to human support. Order ID is 12345."
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Here is a draft of the email I have prepared to escalate this issue to the human support team:

Subject: Customer Support Escalation - Order 12345

Dear Human Support Team,

A customer needs assistance with the following issue:

The order arrived damaged.

Order ID: 12345

Please review this issue and contact the customer with the appropriate resolution.

Regards,
AI Customer Support Agent


In [91]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my order ID?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

I'm sorry, I don't have access to your order ID. Please provide it to me and I can help you further.


In [92]:
config = {
    "configurable": {
        "thread_id": "memory-test-001"
    }
}

response1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "My order ID is 12345. Please remember it."}
        ]
    },
    config=config
)

print(response1["messages"][-1].content)

Thank you for providing your order ID. I have noted it down and will refer to it if needed.


In [93]:

config = {
    "configurable": {
        "thread_id": "memory-test-001"
    }
}

response2 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is my order ID? Reply with the exact number I gave you."
            }
        ]
    },
    config=config
)

print(response2["messages"][-1].content)


Your order ID is 12345.


In [94]:
system_prompt="""
You are a helpful AI customer support agent.

1. Use knowledge_base_search for FAQ questions.
2. Use order_status_lookup for order status questions.
3. Use escalate_to_human for complaints requiring human assistance.
4. Never invent order information.
5. Give polite, clear answers.
6. Remember details from earlier messages in the same conversation thread.
7. Use previous messages to answer follow-up questions.
8. If the customer already provided an order ID in the conversation,
   use that ID when answering follow-up questions.
9. Ask for the order ID only if it is not available in the conversation history.
"""

In [95]:
history = agent.get_state(config)

for message in history.values.get("messages", []):
    print(message.type, ":", message.content)

human : My order ID is 12345. Please remember it.
ai : Thank you for providing your order ID. I have noted it down and will refer to it if needed.
human : What is my order ID? Reply with the exact number I gave you.
ai : Your order ID is 12345.


In [96]:
print("===== PROJECT DEMONSTRATION =====")

print("LLM:", type(llm).__name__)
print("Vector store:", type(vector_store).__name__)
print("Agent:", type(agent).__name__)

print("\nTools:")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")

print("\nProject setup check completed!")

===== PROJECT DEMONSTRATION =====
LLM: ChatCohere
Vector store: FAISS
Agent: CompiledStateGraph

Tools:
- knowledge_base_search: Search the customer support FAQ knowledge base for an answer.
- order_status_lookup: Look up the status and expected delivery date of a customer order.
- escalate_to_human: Draft an escalation email for a customer issue that needs human support.

Project setup check completed!


In [98]:

demo_questions = [
    "How can I reset my password?",
    "What is your refund policy?",
    "What is the status of order 12345?",
    "My order arrived damaged. I need human support."
]

for question in demo_questions:
    print("\n" + "=" * 60)
    print("Customer:", question)

    response = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ]
        },
        config={
            "configurable": {
                "thread_id": "final-demo"
            }
        }
    )

    print("Agent:", response["messages"][-1].content)



Customer: How can I reset my password?
Agent: To reset your password, click "Forgot Password" on the login page. You will then receive a password reset link on your registered email address. The reset link is valid for 30 minutes.

Customer: What is your refund policy?
Agent: Refunds are normally processed within 5 to 7 business days after the returned product is received. The refund will be sent to the original payment method.

Customer: What is the status of order 12345?
Agent: Order 12345 has been shipped and is expected to arrive on 10 October 2026.

Customer: My order arrived damaged. I need human support.
Agent: Here is an email that has been drafted for a human support agent:

---

**Subject: Customer Support Escalation - Order 12345**

Dear Human Support Team,

A customer needs assistance with the following issue:

My order arrived damaged. I need human support.

Order ID: 12345

Please review this issue and contact the customer with the appropriate resolution.

Regards,
AI Cu